## nationality

In [1]:
import pandas as pd
import re
import json
from difflib import get_close_matches

c:\Users\sumse\miniconda3\envs\data_quality\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\sumse\miniconda3\envs\data_quality\lib\site-packages\numpy\.libs\libopenblas_v0.3.21-gcc_8_3_0.dll
c:\Users\sumse\miniconda3\envs\data_quality\lib\site-packages\numpy\.libs\libopenblas_v0.3.23-gcc_8_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [2]:
# import data

data_nat = pd.read_csv(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\nationality\1_customer_comp_nationality.csv" , encoding = 'utf8')

data_nat.shape

(300, 7)

In [3]:
# import dictionary of education(json)

with open(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\nationality\nationality_pattern.json",encoding='utf-8')as fin:
    pattern_dict = json.load(fin)

In [4]:
pd.set_option('display.max_rows', None)

In [5]:
data_nat.head(5)

,subject,no,id,case,detail case,flag,nationality
0,nationality,1,nationality-country_name_en-1,country_name_en,normal,good,Afghanistan
1,nationality,2,nationality-country_name_en-2,country_name_en,normal,good,Albania
2,nationality,3,nationality-country_name_en-3,country_name_en,normal,good,Algeria
3,nationality,4,nationality-country_name_en-4,country_name_en,normal,good,Andorra
4,nationality,5,nationality-country_name_en-5,country_name_en,normal,good,Angola


In [6]:
data_nat.tail(5)

,subject,no,id,case,detail case,flag,nationality
295,nationality,296,nationality-country_name_th-296,country_name_th,spell,bad,ซาอุดิอาไรเบีย
296,nationality,297,nationality-country_name_th-297,country_name_th,spell,bad,ฟิลิบปิน
297,nationality,298,nationality-country_name_th-298,country_name_th,spell,bad,เนเธ้อแลน
298,nationality,299,nationality-country_name_th-299,country_name_th,spell,bad,เม็ตสิโก
299,nationality,300,nationality-country_name_th-300,country_name_th,spell,bad,ไท


In [7]:
# choose only english letters
# remove . and _
# make the data to title case
def take_eng(text_input) :
    try :
        if re.search(r'[a-zA-Z]{3,}$' , text_input):
            text_input = text_input.replace('_' , '').replace('.' , '').title()
            return text_input
    except AttributeError :
        return None

# choose only thai letters
# remove . and _ and white space
def take_thai(text_input) :
    try :
        if re.search(r'[ก-๛]+' , text_input) :
            text_input = text_input.replace('_' , '').replace(' ' , '').replace('.' , '')
            return text_input
    except AttributeError :
        return None

# choose only country code
# remove . and _ and - and white space
# make the data to upper case
def country_code(text_input) :
    try :
        if re.search(r'^[a-zA-Z]{2}$' , text_input):
            text_input = text_input.upper().replace('_' , '').replace(' ' , '').replace('.' , '').replace('-' , '')
            return text_input
    except AttributeError :
        return None
    


# create standard format
# conutrycode and thai letter
# for data didn't be in dictionary find similar word of key and data
def std_nat(text_input, case):
    try:
        if text_input in pattern_dict[case]:
            result = pattern_dict[case][text_input]
            return result
        else:
            similar_keys = get_close_matches(text_input, pattern_dict[case].keys(), n=1, cutoff=0.7)
            if similar_keys:
                value = pattern_dict[case][similar_keys[0]]
                return value
            else:
                return None
    except KeyError:
        return None
    

# create standard format
# english letter
# for data didn't be in dictionary find similar word of key and data
def std_nat_en(text_input, name_en, person, nationality):
    try:
        typ = [name_en, person, nationality]
        
        for case in typ:
            if case in pattern_dict and text_input in pattern_dict[case]:
                result = pattern_dict[case][text_input]
                return result
            else:
                if case in pattern_dict:
                    similar_keys = get_close_matches(text_input, pattern_dict[case].keys(), n=1, cutoff=0.7)
                    if similar_keys:
                        value = pattern_dict[case][similar_keys[0]]
                        return value
        
        return None
    except KeyError:
        return None
    
# last result
def result(text_input, standard):
    if isinstance(text_input , str) and standard == None :
        return text_input , 0
    
    elif text_input == standard :
        return text_input , 1
    
    elif text_input != standard :
        return standard , 2
    
    elif text_input == None and standard == None :
        return text_input , 3

In [8]:
# choose only english letters
# remove . and _
# make the data to title case
def take_eng(text_input) :
    try :
        if re.search(r'[a-zA-Z]{3,}$' , text_input):
            text_input = text_input.replace('_' , '').replace('.' , '').title()
            return text_input
    except AttributeError :
        return None
    
    
    
# create column for tranform data in english
data_nat['df_eng'] = data_nat['nationality'].apply(lambda nationality :  take_eng(nationality))

In [9]:
# choose only thai letters
# remove . and _ and white space
def take_thai(text_input) :
    try :
        if re.search(r'[ก-๛]+' , text_input) :
            text_input = text_input.replace('_' , '').replace(' ' , '').replace('.' , '')
            return text_input
    except AttributeError :
        return None
    
    
    
# create column for tranform data in thai
data_nat['df_thai'] = data_nat['nationality'].apply(lambda nationality :  take_thai(nationality))

In [10]:
# choose only country code
# remove . and _ and - and white space
# make the data to upper case
def country_code(text_input) :
    try :
        if re.search(r'^[a-zA-Z]{2}$' , text_input):
            text_input = text_input.upper().replace('_' , '').replace(' ' , '').replace('.' , '').replace('-' , '')
            return text_input
    except AttributeError :
        return None
    
    
    
# create column for tranform data of country code
data_nat['df_countrycode'] = data_nat['nationality'].apply(lambda nationality :  country_code(nationality))

In [11]:
nationality_en = data_nat[data_nat['df_eng'].notnull()]
nationality_th = data_nat[data_nat['df_thai'].notnull()]
nationality_countrycode = data_nat[data_nat['df_countrycode'].notnull()]

In [12]:
# create standard format
# english letter
# for data didn't be in dictionary find similar word of key and data
def std_nat_en(text_input, name_en, person, nationality):
    try:
        typ = [name_en, person, nationality]
        
        for case in typ:
            if case in pattern_dict and text_input in pattern_dict[case]:
                result = pattern_dict[case][text_input]
                return result
            else:
                if case in pattern_dict:
                    similar_keys = get_close_matches(text_input, pattern_dict[case].keys(), n=1, cutoff=0.7)
                    if similar_keys:
                        value = pattern_dict[case][similar_keys[0]]
                        return value
        
        return None
    except KeyError:
        return None
    
    
nationality_en['standard_nationality'] = nationality_en['df_eng'].apply(lambda df_eng :  std_nat_en(df_eng , 'country_nationality', 'person_nationality', 'nationality_nationality'))

C:\Users\sumse\AppData\Local\Temp\ipykernel_29360\2513984106.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nationality_en['standard_nationality'] = nationality_en['df_eng'].apply(lambda df_eng :  std_nat_en(df_eng , 'country_nationality', 'person_nationality', 'nationality_nationality'))


In [13]:
# create standard format
# conutrycode and thai letter
# for data didn't be in dictionary find similar word of key and data
def std_nat(text_input, case):
    try:
        if text_input in pattern_dict[case]:
            result = pattern_dict[case][text_input]
            return result
        else:
            similar_keys = get_close_matches(text_input, pattern_dict[case].keys(), n=1, cutoff=0.7)
            if similar_keys:
                value = pattern_dict[case][similar_keys[0]]
                return value
            else:
                return None
    except KeyError:
        return None
    
    
    
# create column for convert from country code to country and convert from country to nationality
nationality_countrycode['standard_nationality'] = nationality_countrycode['df_countrycode'].apply(lambda df_countrycode :  std_nat(df_countrycode , 'contrycode_country'))
nationality_countrycode['standard_nationality'] = nationality_countrycode['standard_nationality'].apply(lambda standard_nationality: std_nat(standard_nationality, 'country_nationality') if standard_nationality is not None else None)

C:\Users\sumse\AppData\Local\Temp\ipykernel_29360\3071300866.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nationality_countrycode['standard_nationality'] = nationality_countrycode['df_countrycode'].apply(lambda df_countrycode :  std_nat(df_countrycode , 'contrycode_country'))
C:\Users\sumse\AppData\Local\Temp\ipykernel_29360\3071300866.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nationality_countrycode['standard_nationality'] = nationality_countrycode['standard_nationality'].apply(lambda st

In [14]:
# create column for convert from country in thai to country in english and convert from country in english to nationality
nationality_th['standard_nationality'] = nationality_th['df_thai'].apply(lambda df_thai :  std_nat(df_thai , 'country_thai_eng'))
nationality_th['standard_nationality'] = nationality_th['standard_nationality'].apply(lambda standard_nationality :  std_nat(standard_nationality , 'country_nationality')if standard_nationality is not None else None)

C:\Users\sumse\AppData\Local\Temp\ipykernel_29360\3313044331.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nationality_th['standard_nationality'] = nationality_th['df_thai'].apply(lambda df_thai :  std_nat(df_thai , 'country_thai_eng'))
C:\Users\sumse\AppData\Local\Temp\ipykernel_29360\3313044331.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nationality_th['standard_nationality'] = nationality_th['standard_nationality'].apply(lambda standard_nationality :  std_nat(standard_nationality , 'country

In [15]:
# combined data frame of all cases
nationality = pd.concat([nationality_en, nationality_countrycode, nationality_th])
nationality['nationality_transform'] = nationality['df_eng'].combine_first(nationality['df_countrycode']).combine_first(nationality['df_thai'])


del nationality_en, nationality_countrycode, nationality_th

In [16]:
# last result
def result(text_input, standard):
    if isinstance(text_input , str) and standard == None :
        return text_input , 0
    
    elif text_input == standard :
        return text_input , 1
    
    elif text_input != standard :
        return standard , 2
    
    elif text_input == None and standard == None :
        return text_input , 3
    
    
    
nationality[['std_output' , 'status']] = nationality.apply(lambda row: pd.Series(result(row['nationality'], row['standard_nationality'])), axis=1)

In [18]:
selected_cols = ['id','nationality','standard_nationality','std_output' ,'status']

nationality = nationality[selected_cols]

In [19]:
nationality.tail(5)

,id,nationality,standard_nationality,std_output,status
295,nationality-country_name_th-296,ซาอุดิอาไรเบีย,Saudi Arabian,Saudi Arabian,2
296,nationality-country_name_th-297,ฟิลิบปิน,Philippine,Philippine,2
297,nationality-country_name_th-298,เนเธ้อแลน,Dutch,Dutch,2
298,nationality-country_name_th-299,เม็ตสิโก,Mexican,Mexican,2
299,nationality-country_name_th-300,ไท,Thai,Thai,2
